# 17 — Automatic Prompt Optimization (APO) and DSPy

## Scenario
We have reached the culmination of prompt engineering: **Prompt Programming**. 

Instead of manually tweaking words in a prompt until it passes an evaluation, we can use a second LLM (an "Optimizer") to read the evaluation failures and *automatically rewrite the prompt for us*.

This is the core mechanic behind frameworks like **DSPy**. In this notebook, we will simulate this mechanic using vanilla Python to understand how it works under the hood.

In [ ]:
import os
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# 1. The Dataset and Failing Baseline
dataset = [
    {"input": "Extract the city from: I live in San Francisco, CA.", "expected": "San Francisco"},
]

class ExtractionResult(BaseModel):
    city: str

baseline_prompt = "Extract the city."

def run_task(prompt, text):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=f"{prompt}\nText: {text}",
        config=types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
            response_schema=ExtractionResult,
        )
    )
    return ExtractionResult.model_validate_json(response.text).city

print("--- BASELINE EVALUATION ---")
test_case = dataset[0]
actual_output = run_task(baseline_prompt, test_case['input'])

if actual_output != test_case['expected']:
    print(f"FAIL: Expected '{test_case['expected']}', Got '{actual_output}'")


## Step 1: The Optimizer LLM

Because the baseline failed (it probably extracted "San Francisco, CA" instead of just "San Francisco"), we need to fix the prompt.

Instead of writing the fix ourselves, we ask an Optimizer LLM to write a new prompt.

In [ ]:
class OptimizedPrompt(BaseModel):
    new_instructions: str = Field(description="The new, improved instructions for the task.")

optimizer_prompt = f"""\nYou are an expert prompt engineer.\nYour job is to rewrite the instructions for a task so that it passes the failing test case.\n\nCURRENT INSTRUCTIONS: {baseline_prompt}\n\nFAILING TEST CASE:\n- Input: {test_case['input']}\n- Expected Output: {test_case['expected']}\n- Actual Output (Error): {actual_output}\n\nWrite new instructions that are robust and will successfully extract the expected output.\n"""

response_optimizer = client.models.generate_content(
    model=MODEL_ID,
    contents=optimizer_prompt,
    config=types.GenerateContentConfig(
        temperature=0.7,
        response_mime_type="application/json",
        response_schema=OptimizedPrompt,
    )
)

new_prompt = OptimizedPrompt.model_validate_json(response_optimizer.text).new_instructions

print("\n--- OPTIMIZER LLM OUTPUT ---")
print(f"Old Prompt: {baseline_prompt}")
print(f"New Prompt: {new_prompt}")


## Step 2: Testing the Optimized Prompt

We now run the exact same task, but using the `new_prompt` generated by our Optimizer.

In [ ]:
print("\n--- OPTIMIZED EVALUATION ---")
new_actual_output = run_task(new_prompt, test_case['input'])

if new_actual_output == test_case['expected']:
    print(f"SUCCESS: Got exactly '{new_actual_output}'")
else:
    print(f"STILL FAILING: Got '{new_actual_output}'")


## Conclusion: DSPy and the Future

What you just witnessed is the core loop of **Automatic Prompt Optimization (APO)**.

In the real world, you do not write this loop yourself. You use frameworks like **DSPy**.
In DSPy, you:
1. Define a Signature (Input -> Output).
2. Define a Metric (e.g., Exact Match).
3. Provide a Dataset.
4. Call `compile()`.

DSPy handles the Optimizer LLM, iterates over your dataset, generates dozens of candidate prompts, tests them, and outputs the mathematical best prompt for your specific use case. You no longer "engineer" prompts; you compile them.